In [27]:
import pandas as pd

transactions = pd.read_csv(
    "../data/synthetic/transactions.csv"
)

customers = pd.read_csv(
    "../data/synthetic/customers.csv"
)

merchants = pd.read_csv(
    "../data/synthetic/merchants.csv"
)

devices = pd.read_csv(
    "../data/synthetic/devices.csv"
)
profiles = pd.read_csv(

    "../data/synthetic/customer_profiles.csv"

)
transactions.head()

,transaction_id,customer_id,merchant_id,device_id,transaction_time,transaction_type,channel,amount,currency,country,city,status
0,TXN_00000001,CUST_000001,MERCH_000654,DEV_000001,2026-01-01 03:43:54,MERCHANT_PAYMENT,MOBILE_APP,11308.39,NGN,Nigeria,Benin City,SUCCESS
1,TXN_00000002,CUST_000001,MERCH_001591,DEV_000001,2026-01-01 01:05:05,MERCHANT_PAYMENT,POS,15152.03,NGN,Nigeria,Benin City,SUCCESS
2,TXN_00000003,CUST_000001,NaN,DEV_000001,2026-01-01 23:39:41,P2P_TRANSFER,MOBILE_APP,7564.68,NGN,Nigeria,Benin City,SUCCESS
3,TXN_00000004,CUST_000001,MERCH_001588,DEV_000001,2026-01-01 00:14:11,CARD_PAYMENT,WEB,8415.87,NGN,Nigeria,Benin City,SUCCESS
4,TXN_00000005,CUST_000004,MERCH_000903,DEV_000004,2026-01-01 07:50:21,CARD_PAYMENT,MOBILE_APP,19772.85,NGN,Nigeria,Ibadan,SUCCESS


In [28]:
transactions.shape

(5076856, 12)

In [29]:
transactions["transaction_type"].value_counts(
    normalize=True
    ) * 100

transaction_type
P2P_TRANSFER        35.004085
MERCHANT_PAYMENT    34.997014
CARD_PAYMENT        20.010140
CASH_WITHDRAWAL      9.988761
Name: proportion, dtype: float64

In [30]:
transactions["channel"].value_counts(
    normalize=True
) * 100

channel
MOBILE_APP    38.994409
POS           26.262002
WEB           17.486866
USSD           8.759279
ATM            8.497444
Name: proportion, dtype: float64

In [31]:
transactions["status"].value_counts(
    normalize=True
) * 100

status
SUCCESS     93.986751
FAILED       5.019484
REVERSED     0.993765
Name: proportion, dtype: float64

In [32]:
pd.crosstab(
    transactions["country"],
    transactions["currency"]
)

currency,GHS,NGN
country,,
Ghana,1819985,0
Nigeria,0,3256871


In [33]:
transactions["amount"].describe()

count    5.076856e+06
mean     3.158341e+04
std      5.725707e+04
min      1.000000e+02
25%      1.977900e+02
50%      1.172021e+04
75%      3.788689e+04
max      2.189050e+06
Name: amount, dtype: float64

In [34]:
transactions["transaction_time"] = pd.to_datetime(
    transactions["transaction_time"]
)

transactions["transaction_time"].min(), transactions[
    "transaction_time"
].max()

(Timestamp('2026-01-01 00:00:00'), Timestamp('2026-08-31 23:59:52'))

In [35]:
transactions["customer_id"].isin(
    customers["customer_id"]
).all()

np.True_

In [36]:
transactions["merchant_id"].isin(
    merchants["merchant_id"]
).all()

np.False_

In [37]:
transactions["device_id"].isin(
    devices["device_id"]
).all()

np.True_

In [38]:
customer_devices = pd.read_csv(
    "../data/synthetic/customer_devices.csv"
)

In [39]:
valid_customer_device_pairs = set(
    zip(
        customer_devices["customer_id"],
        customer_devices["device_id"],
    )
)

transaction_pairs = list(
    zip(
        transactions["customer_id"],
        transactions["device_id"],
    )
)

all(
    pair in valid_customer_device_pairs
    for pair in transaction_pairs
)

True

In [40]:
transactions["amount"].describe()

count    5.076856e+06
mean     3.158341e+04
std      5.725707e+04
min      1.000000e+02
25%      1.977900e+02
50%      1.172021e+04
75%      3.788689e+04
max      2.189050e+06
Name: amount, dtype: float64

In [41]:
transactions.groupby(
    ["currency", "transaction_type"]
)["amount"].agg(
    ["count", "mean", "median", "min", "max"]
).round(2)

count      mean    median     min         max
currency transaction_type                                                 
GHS      CARD_PAYMENT       364240    232.36    110.83  100.00     6957.69
         CASH_WITHDRAWAL    181465    242.41    110.94  100.00     7198.59
         MERCHANT_PAYMENT   637097    236.73    110.73  100.00     8571.90
         P2P_TRANSFER       637183    247.45    111.33  100.00    11187.45
NGN      CARD_PAYMENT       651646  47533.23  27798.10  400.77  1521831.46
         CASH_WITHDRAWAL    325650  49468.66  27781.26  311.30  1796663.69
         MERCHANT_PAYMENT  1139651  48407.63  27730.29  328.26  2189049.84
         P2P_TRANSFER      1139924  50578.10  27663.33  248.71  2040683.67

In [42]:
transactions.groupby(
    "currency"
)["amount"].quantile(
    [0.50, 0.75, 0.90, 0.95, 0.99]
).round(2)

currency      
GHS       0.50       110.97
          0.75       238.80
          0.90       509.42
          0.95       792.47
          0.99      1624.98
NGN       0.50     27727.64
          0.75     58684.08
          0.90    113624.56
          0.95    164785.88
          0.99    317653.87
Name: amount, dtype: float64

In [43]:
pd.crosstab(
    transactions["transaction_type"],
    transactions["channel"],
    normalize="index",
) * 100

channel,ATM,MOBILE_APP,POS,USSD,WEB
transaction_type,,,,,
CARD_PAYMENT,0.000000,19.976257,45.004853,0.000000,35.018890
CASH_WITHDRAWAL,85.070053,0.000000,14.929947,0.000000,0.000000
MERCHANT_PAYMENT,0.000000,39.984736,45.047131,0.000000,14.968133
P2P_TRANSFER,0.000000,60.003478,0.000000,25.023592,14.972931


In [44]:
transactions[
    transactions["currency"] == "GHS"
]["amount"].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99])

count    1.819985e+06
mean     2.401749e+02
std      3.158390e+02
min      1.000000e+02
50%      1.109700e+02
75%      2.388000e+02
90%      5.094200e+02
95%      7.924680e+02
99%      1.624976e+03
max      1.118745e+04
Name: amount, dtype: float64

In [45]:
(
    transactions[
        transactions["currency"] == "GHS"
    ]["amount"] == 100
).mean() * 100

np.float64(46.11702843704756)

In [46]:
validation = transactions.merge(
    profiles[
        [
            "customer_id",
            "activity_profile",
            "typical_transaction_amount",
            "currency",
        ]
    ],
    on="customer_id",
    how="left",
)

In [47]:
validation.groupby(
    "activity_profile"
)["amount"].median()

activity_profile
HIGH      30715.34
LOW        4301.36
MEDIUM    11679.41
Name: amount, dtype: float64

In [48]:
import numpy as np

np.random.seed(42)

daily_transactions = np.random.poisson(
    lam=3,
    size=30
)

daily_transactions

array([4, 1, 3, 3, 2, 3, 2, 3, 0, 2, 4, 2, 1, 3, 4, 4, 1, 2, 3, 2, 3, 4,
       1, 3, 3, 3, 5, 0, 1, 4])

In [49]:
daily_transactions.mean()

np.float64(2.533333333333333)

In [50]:
daily_transactions.min(), daily_transactions.max()

(np.int64(0), np.int64(5))

In [51]:
transactions_with_profiles.groupby(
    "activity_profile"
).size()

NameError: name 'transactions_with_profiles' is not defined